# 03 - Preparação dos Dados

**Projeto:** Análise do Mercado Imobiliário Português  
**Fase CRISP-DM:** Data Preparation  
**Objetivo:** transformar o dataset processado inicial numa versão preparada, auditável e pronta para análise avançada, feature engineering e modelação futura.  
**Última alteração:** 18/06/2026

Este notebook dá continuidade às fases de inicialização e compreensão dos dados. A preparação aqui feita é conservadora: corrige problemas objetivos, sinaliza riscos e evita remover valores extremos sem análise contextual.

Entradas esperadas:
- `data/processed/portugal_listings_initial_clean.csv`, quando disponível localmente.
- Ficheiro processado inicial no GitHub, quando existir.
- `data/raw/portugal_listinigs.csv` ou ficheiro raw no GitHub como alternativa.

Saída esperada:
- `data/processed/portugal_listings_prepared.csv`, em ambiente local.
- `data/processed/portugal_listings_prepared.csv.gz`, versão comprimida recomendada para GitHub.
- `/kaggle/working/portugal_listings_prepared.csv` e `/kaggle/working/portugal_listings_prepared.csv.gz`, em ambiente Kaggle.

Nota metodológica:
- Esta fase prepara dados, mas não faz encoding final, imputação definitiva nem seleção final de variáveis para modelo. Essas decisões pertencem às fases de feature engineering e modelação.


## Decisões de Preparação

As decisões aplicadas neste notebook seguem três princípios:

- preservar `data/raw/` sem alterações;
- aplicar apenas regras objetivas de domínio;
- sinalizar incerteza, valores em falta e outliers em vez de os remover automaticamente.

Regras principais:

| Tema | Decisão |
|---|---|
| Duplicados | Remover duplicados exatos na versão preparada |
| Variável alvo | Remover linhas sem `price` |
| Áreas | Converter áreas não positivas para valores em falta |
| Contagens | Converter contagens negativas para valores em falta |
| Ano de construção | Tratar anos fora de `[1900, 2026]` como inválidos |
| Missing values | Criar indicadores de ausência para variáveis relevantes |
| Outliers | Criar flags por IQR; não remover observações nesta fase |
| Fuga de informação | Manter `price_m2` apenas para análise, não como preditor direto de `price` |


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

GITHUB_BASE_URL = "https://raw.githubusercontent.com/LuiscnFigueira/portugal-real-estate-market-analysis/main"
INITIAL_PROCESSED_FILENAME = "portugal_listings_initial_clean.csv"
RAW_FILENAME = "portugal_listinigs.csv"
PREPARED_FILENAME = "portugal_listings_prepared.csv"
PREPARED_COMPRESSED_FILENAME = "portugal_listings_prepared.csv.gz"
REFERENCE_YEAR = 2026


def find_project_root(start=None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        markers = [candidate / "README.md", candidate / "requirements.txt", candidate / "data"]
        if all(marker.exists() for marker in markers):
            return candidate
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    return current


PROJECT_ROOT = find_project_root()
IS_KAGGLE = Path("/kaggle/working").exists()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LOCAL_PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / INITIAL_PROCESSED_FILENAME
LOCAL_RAW_PATH = PROJECT_ROOT / "data" / "raw" / RAW_FILENAME
KAGGLE_PREPARED_PATH = Path("/kaggle/working") / PREPARED_FILENAME
LOCAL_PREPARED_PATH = PROJECT_ROOT / "data" / "processed" / PREPARED_FILENAME
PREPARED_DATA_PATH = KAGGLE_PREPARED_PATH if IS_KAGGLE else LOCAL_PREPARED_PATH
PREPARED_COMPRESSED_PATH = PREPARED_DATA_PATH.with_name(PREPARED_COMPRESSED_FILENAME)

GITHUB_PROCESSED_URL = f"{GITHUB_BASE_URL}/data/processed/{INITIAL_PROCESSED_FILENAME}"
GITHUB_RAW_URL = f"{GITHUB_BASE_URL}/data/raw/{RAW_FILENAME}"

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Ambiente Kaggle: {IS_KAGGLE}")
print(f"Saída preparada: {PREPARED_DATA_PATH}")
print(f"Saída comprimida: {PREPARED_COMPRESSED_PATH}")


Raiz do projeto: /kaggle/working
Ambiente Kaggle: True
Saída preparada: /kaggle/working/portugal_listings_prepared.csv
Saída comprimida: /kaggle/working/portugal_listings_prepared.csv.gz


## Funções de Preparação

Sempre que os módulos `src.data.preprocess` e `src.data.prepare` estiverem disponíveis, o notebook usa as funções reutilizáveis do projeto. Em ambiente Kaggle isolado, onde o notebook pode existir sem a pasta `src/`, são criadas funções equivalentes de fallback para manter a execução reproduzível.

Este bloco também garante compatibilidade com duas situações:

- dataset processado inicial, com colunas em `snake_case`;
- dataset raw vindo do GitHub, com colunas originais em PascalCase.


In [2]:
try:
    from src.data.preprocess import build_initial_dataset
    from src.data.prepare import (
        calculate_iqr_outlier_summary,
        prepare_dataset,
    )
    print("Funções carregadas a partir de src.data")
except ModuleNotFoundError:
    print("Módulo src não encontrado. A usar funções locais de fallback.")

    RAW_COLUMN_NAMES = [
        "price", "district", "city", "town", "type", "energy_certificate",
        "gross_area", "total_area", "parking", "has_parking", "floor",
        "construction_year", "energy_efficiency_level", "publish_date",
        "garage", "elevator", "electric_cars_charging", "total_rooms",
        "number_of_bedrooms", "number_of_wc", "conservation_status",
        "living_area", "lot_size", "built_area", "number_of_bathrooms",
    ]

    BOOLEAN_COLUMNS = ["has_parking", "garage", "elevator", "electric_cars_charging"]
    CATEGORICAL_COLUMNS = [
        "district", "city", "town", "type", "energy_certificate", "floor",
        "energy_efficiency_level", "conservation_status",
    ]
    INTEGER_COLUMNS = [
        "parking", "construction_year", "total_rooms", "number_of_bedrooms",
        "number_of_wc", "number_of_bathrooms",
    ]
    AREA_COLUMNS = ["gross_area", "total_area", "living_area", "lot_size", "built_area"]
    COUNT_COLUMNS = ["parking", "total_rooms", "number_of_bedrooms", "number_of_wc", "number_of_bathrooms"]
    MISSING_INDICATOR_COLUMNS = [
        "gross_area", "living_area", "lot_size", "built_area", "construction_year",
        "total_rooms", "number_of_bedrooms", "number_of_wc", "number_of_bathrooms",
        "energy_certificate", "conservation_status",
    ]
    OUTLIER_COLUMNS = [
        "price", "gross_area", "total_area", "living_area", "lot_size", "built_area",
        "total_rooms", "number_of_bathrooms", "price_m2",
    ]

    def rename_raw_columns(df):
        df = df.copy()
        if len(df.columns) != len(RAW_COLUMN_NAMES):
            raise ValueError(
                f"Número inesperado de colunas raw: {len(df.columns)}. "
                f"Esperado: {len(RAW_COLUMN_NAMES)}."
            )
        df.columns = RAW_COLUMN_NAMES
        return df

    def convert_initial_types(df):
        df = df.copy()
        df["publish_date"] = pd.to_datetime(df["publish_date"], errors="coerce")

        boolean_map = {
            "true": True, "false": False, "yes": True, "no": False,
            "sim": True, "não": False, "nao": False,
        }
        for col in BOOLEAN_COLUMNS:
            normalized = df[col].astype("string").str.lower().str.strip()
            df[col] = normalized.map(boolean_map).astype("boolean")

        for col in CATEGORICAL_COLUMNS:
            df[col] = df[col].astype("category")

        for col in INTEGER_COLUMNS:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

        return df

    def add_initial_features(df, reference_year=REFERENCE_YEAR):
        df = df.copy()
        df["price_m2"] = df["price"] / df["total_area"]
        df["price_m2"] = df["price_m2"].replace([np.inf, -np.inf], np.nan)
        df.loc[df["price_m2"] <= 0, "price_m2"] = np.nan
        df["property_age"] = reference_year - df["construction_year"]
        df.loc[df["property_age"] < 0, "property_age"] = pd.NA
        df["publish_year"] = df["publish_date"].dt.year
        df["publish_month"] = df["publish_date"].dt.month
        return df

    def build_initial_dataset(df, reference_year=REFERENCE_YEAR):
        df = rename_raw_columns(df)
        df = convert_initial_types(df)
        df = enforce_domain_rules(df, reference_year=reference_year)
        df = add_initial_features(df, reference_year=reference_year)
        df = df.drop_duplicates()
        df = df.dropna(subset=["price"])
        return df.reset_index(drop=True)

    def normalize_text_values(df):
        df = df.copy()
        text_columns = df.select_dtypes(include=["object", "category", "string"]).columns
        for col in text_columns:
            normalized = df[col].astype("string").str.strip()
            df[col] = normalized.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        return df

    def enforce_domain_rules(df, reference_year=REFERENCE_YEAR):
        df = df.copy()
        if "price" in df.columns:
            df.loc[df["price"] <= 0, "price"] = np.nan
        for col in AREA_COLUMNS:
            if col in df.columns:
                df.loc[df[col] <= 0, col] = np.nan
        for col in COUNT_COLUMNS:
            if col in df.columns:
                df.loc[df[col] < 0, col] = np.nan
        if "construction_year" in df.columns:
            invalid_year = (df["construction_year"] < 1900) | (df["construction_year"] > reference_year)
            df.loc[invalid_year, "construction_year"] = np.nan
        if {"construction_year", "property_age"}.issubset(df.columns):
            df["property_age"] = reference_year - df["construction_year"]
            df.loc[df["property_age"] < 0, "property_age"] = np.nan
        if {"price", "total_area", "price_m2"}.issubset(df.columns):
            df["price_m2"] = df["price"] / df["total_area"]
            df["price_m2"] = df["price_m2"].replace([np.inf, -np.inf], np.nan)
            df.loc[df["price_m2"] <= 0, "price_m2"] = np.nan
        return df

    def add_missing_indicators(df, columns=None):
        df = df.copy()
        for col in columns or MISSING_INDICATOR_COLUMNS:
            if col in df.columns:
                df[f"{col}_missing"] = df[col].isna()
        return df

    def calculate_iqr_outlier_summary(df, columns=None):
        rows = []
        for col in columns or OUTLIER_COLUMNS:
            if col not in df.columns:
                continue
            series = pd.to_numeric(df[col], errors="coerce").dropna()
            if series.empty:
                continue
            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            count = int(((series < lower) | (series > upper)).sum())
            rows.append({
                "column": col, "q1": q1, "q3": q3, "iqr": iqr,
                "lower_bound": lower, "upper_bound": upper,
                "outlier_count": count, "outlier_share": count / len(series),
            })
        return pd.DataFrame(rows)

    def add_iqr_outlier_flags(df, columns=None):
        df = df.copy()
        summary = calculate_iqr_outlier_summary(df, columns=columns)
        for row in summary.itertuples(index=False):
            df[f"{row.column}_iqr_outlier"] = (df[row.column] < row.lower_bound) | (df[row.column] > row.upper_bound)
            df[f"{row.column}_iqr_outlier"] = df[f"{row.column}_iqr_outlier"].fillna(False)
        return df

    def prepare_dataset(df, reference_year=REFERENCE_YEAR):
        prepared = normalize_text_values(df)
        prepared = enforce_domain_rules(prepared, reference_year=reference_year)
        prepared = prepared.drop_duplicates()
        prepared = prepared.dropna(subset=["price"])
        prepared = add_missing_indicators(prepared)
        prepared = add_iqr_outlier_flags(prepared)
        return prepared.reset_index(drop=True)


def standardize_input_dataset(df, reference_year=REFERENCE_YEAR):
    """Garante que o notebook aceita dados processados ou raw."""
    if "price" in df.columns:
        standardized = df.copy()
        if "publish_date" in standardized.columns:
            standardized["publish_date"] = pd.to_datetime(standardized["publish_date"], errors="coerce")
        if {"price", "total_area"}.issubset(standardized.columns) and "price_m2" not in standardized.columns:
            standardized["price_m2"] = standardized["price"] / standardized["total_area"]
            standardized["price_m2"] = standardized["price_m2"].replace([np.inf, -np.inf], np.nan)
        if {"construction_year"}.issubset(standardized.columns) and "property_age" not in standardized.columns:
            standardized["property_age"] = reference_year - standardized["construction_year"]
        if "publish_date" in standardized.columns:
            if "publish_year" not in standardized.columns:
                standardized["publish_year"] = standardized["publish_date"].dt.year
            if "publish_month" not in standardized.columns:
                standardized["publish_month"] = standardized["publish_date"].dt.month
        return standardized, "processed"

    if "Price" in df.columns:
        return build_initial_dataset(df, reference_year=reference_year), "raw"

    raise ValueError(
        "Formato de dataset não reconhecido. Esperava colunas processadas "
        "como 'price' ou colunas raw como 'Price'."
    )


Módulo src não encontrado. A usar funções locais de fallback.


## Carregamento dos Dados

O notebook tenta carregar primeiro o dataset processado inicial. Se essa versão ainda não existir no ambiente, usa as fontes alternativas definidas para Kaggle e GitHub.

No Kaggle, isto permite dois cenários:

- usar um dataset anexado ao notebook em `/kaggle/input`;
- carregar diretamente do GitHub através dos URLs raw definidos no início.

Se a fonte disponível for o ficheiro raw, o notebook aplica automaticamente a preparação inicial antes de avançar para a fase 03.


In [3]:
def kaggle_candidates(filename: str) -> list[Path]:
    base = Path("/kaggle/input")
    if not base.exists():
        return []
    return sorted(base.glob(f"**/{filename}"))


def read_first_available_csv(sources):
    errors = []
    for source in sources:
        if isinstance(source, Path) and not source.exists():
            continue
        try:
            return pd.read_csv(source, low_memory=False), source
        except Exception as exc:
            errors.append(f"{source}: {exc}")
    raise FileNotFoundError(
        "Não foi possível carregar dados. Fontes testadas:"
        + "\n"
        + "\n".join(map(str, sources + errors))
    )


sources = [
    LOCAL_PROCESSED_PATH,
    *kaggle_candidates(INITIAL_PROCESSED_FILENAME),
    GITHUB_PROCESSED_URL,
    LOCAL_RAW_PATH,
    *kaggle_candidates(RAW_FILENAME),
    GITHUB_RAW_URL,
]

loaded_df, data_source = read_first_available_csv(sources)
df, input_format = standardize_input_dataset(loaded_df, reference_year=REFERENCE_YEAR)

print(f"Fonte usada: {data_source}")
print(f"Formato carregado: {input_format}")
print(f"Dataset disponível para preparação: {df.shape[0]:,} linhas e {df.shape[1]:,} colunas")
display(df.head())


Fonte usada: https://raw.githubusercontent.com/LuiscnFigueira/portugal-real-estate-market-analysis/main/data/processed/portugal_listings_initial_clean.csv
Formato carregado: processed
Dataset disponível para preparação: 126,242 linhas e 29 colunas


,price,district,city,town,type,energy_certificate,gross_area,total_area,parking,has_parking,floor,construction_year,energy_efficiency_level,publish_date,garage,elevator,electric_cars_charging,total_rooms,number_of_bedrooms,number_of_wc,conservation_status,living_area,lot_size,built_area,number_of_bathrooms,price_m2,property_age,publish_year,publish_month
0,"780,000.00",Vila Real,Valpaços,Carrazedo de Montenegro e Curros,Farm,NC,200.00,"552,450.00",0.00,False,NaN,NaN,NaN,NaT,NaN,False,NaN,NaN,NaN,NaN,NaN,120.00,NaN,NaN,0.00,1.41,NaN,NaN,NaN
1,"223,000.00",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,81.00,1.00,True,Ground Floor,NaN,NaN,NaT,NaN,True,NaN,2.00,NaN,NaN,NaN,81.00,NaN,NaN,2.00,"2,753.09",NaN,NaN,NaN
2,"228,000.00",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,108.00,1.00,True,Ground Floor,NaN,NaN,NaT,NaN,True,NaN,2.00,NaN,NaN,NaN,108.00,NaN,NaN,2.00,"2,111.11",NaN,NaN,NaN
3,"250,000.00",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.00,1.00,True,1st Floor,NaN,NaN,NaT,NaN,True,NaN,2.00,NaN,NaN,NaN,114.00,NaN,NaN,0.00,"2,192.98",NaN,NaN,NaN
4,"250,000.00",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.00,1.00,True,2nd Floor,NaN,NaN,NaT,NaN,True,NaN,2.00,NaN,NaN,NaN,114.00,NaN,NaN,2.00,"2,192.98",NaN,NaN,NaN


## Diagnóstico Antes da Preparação

Antes de aplicar novas regras, é importante medir o estado do dataset de entrada. Isto permite justificar o impacto da preparação e verificar se a fase anterior foi executada corretamente.


In [4]:
baseline_summary = pd.DataFrame(
    {
        "indicador": [
            "linhas",
            "colunas",
            "duplicados_exatos",
            "price_missing",
            "total_area_missing",
            "price_m2_missing" if "price_m2" in df.columns else "price_m2_indisponivel",
        ],
        "valor": [
            df.shape[0],
            df.shape[1],
            int(df.duplicated().sum()),
            int(df["price"].isna().sum()) if "price" in df.columns else None,
            int(df["total_area"].isna().sum()) if "total_area" in df.columns else None,
            int(df["price_m2"].isna().sum()) if "price_m2" in df.columns else None,
        ],
    }
)

display(baseline_summary)


,indicador,valor
0,linhas,126242
1,colunas,29
2,duplicados_exatos,0
3,price_missing,0
4,total_area_missing,8671
5,price_m2_missing,8671


## Valores em Falta

Nesta fase não será aplicada imputação definitiva. Em vez disso, são criados indicadores de ausência para variáveis relevantes. Esta abordagem preserva informação útil e evita tomar decisões prematuras antes da modelação.


In [5]:
missing_before = (
    pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_share": df.isna().mean().values,
        }
    )
    .sort_values("missing_share", ascending=False)
    .reset_index(drop=True)
)

display(missing_before.head(15))


,column,missing_count,missing_share
0,conservation_status,108263,0.86
1,built_area,101711,0.81
2,gross_area,100574,0.80
3,floor,100224,0.79
4,publish_month,98770,0.78
5,publish_date,98770,0.78
6,publish_year,98770,0.78
7,lot_size,90113,0.71
8,number_of_bedrooms,82586,0.65
9,number_of_wc,73208,0.58


## Outliers

Os outliers são sinalizados com a regra IQR para diagnóstico. Nesta fase não são removidos, porque alguns imóveis extremos podem ser legítimos e dependem do tipo, localização e área.


In [6]:
outlier_summary_before = calculate_iqr_outlier_summary(df)
if not outlier_summary_before.empty:
    display(
        outlier_summary_before.assign(
            outlier_share=lambda data: data["outlier_share"].mul(100).round(2)
        ).sort_values("outlier_count", ascending=False)
    )
else:
    print("Não foram encontradas colunas numéricas suficientes para diagnóstico IQR.")


,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_share
2,total_area,92.00,520.00,428.00,-550.00,"1,162.00",20084,17.08
3,living_area,80.00,206.00,126.00,-109.00,395.00,11188,11.48
0,price,"85,000.00","390,000.00","305,000.00","-372,500.00","847,500.00",10582,8.38
4,lot_size,276.00,"3,000.00","2,724.00","-3,810.00","7,086.00",5052,13.98
8,price_m2,281.48,"2,933.33","2,651.85","-3,696.30","6,911.11",4226,3.59
1,gross_area,102.00,300.00,198.00,-195.00,597.00,2804,10.92
5,built_area,105.00,300.00,195.00,-187.50,592.50,2789,11.37
6,total_rooms,2.00,4.00,2.00,-1.00,7.00,2722,3.97
7,number_of_bathrooms,0.00,2.00,2.00,-3.00,5.00,2579,2.15


## Aplicação da Preparação

A preparação gera uma nova versão do dataset. As regras aplicadas são rastreáveis e não alteram os ficheiros originais.


In [7]:
prepared = prepare_dataset(df, reference_year=REFERENCE_YEAR)

comparison = pd.DataFrame(
    {
        "estado": ["entrada", "preparado"],
        "linhas": [df.shape[0], prepared.shape[0]],
        "colunas": [df.shape[1], prepared.shape[1]],
        "duplicados_exatos": [int(df.duplicated().sum()), int(prepared.duplicated().sum())],
        "price_missing": [int(df["price"].isna().sum()), int(prepared["price"].isna().sum())],
    }
)

display(comparison)
display(prepared.head())


,estado,linhas,colunas,duplicados_exatos,price_missing
0,entrada,126242,29,0,0
1,preparado,126242,49,0,0


,price,district,city,town,type,energy_certificate,gross_area,total_area,parking,has_parking,floor,construction_year,energy_efficiency_level,publish_date,garage,elevator,electric_cars_charging,total_rooms,number_of_bedrooms,number_of_wc,conservation_status,living_area,lot_size,built_area,number_of_bathrooms,price_m2,property_age,publish_year,publish_month,gross_area_missing,living_area_missing,lot_size_missing,built_area_missing,construction_year_missing,total_rooms_missing,number_of_bedrooms_missing,number_of_wc_missing,number_of_bathrooms_missing,energy_certificate_missing,conservation_status_missing,price_iqr_outlier,gross_area_iqr_outlier,total_area_iqr_outlier,living_area_iqr_outlier,lot_size_iqr_outlier,built_area_iqr_outlier,total_rooms_iqr_outlier,number_of_bathrooms_iqr_outlier,price_m2_iqr_outlier
0,"780,000.00",Vila Real,Valpaços,Carrazedo de Montenegro e Curros,Farm,NC,200.00,"552,450.00",0.00,False,<NA>,NaN,<NA>,NaT,<NA>,False,<NA>,NaN,NaN,NaN,<NA>,120.00,NaN,NaN,0.00,1.41,NaN,NaN,NaN,False,False,True,True,True,True,True,True,False,False,True,False,False,True,False,False,False,False,False,False
1,"223,000.00",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,81.00,1.00,True,Ground Floor,NaN,<NA>,NaT,<NA>,True,<NA>,2.00,NaN,NaN,<NA>,81.00,NaN,NaN,2.00,"2,753.09",NaN,NaN,NaN,True,False,True,True,True,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False
2,"228,000.00",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,108.00,1.00,True,Ground Floor,NaN,<NA>,NaT,<NA>,True,<NA>,2.00,NaN,NaN,<NA>,108.00,NaN,NaN,2.00,"2,111.11",NaN,NaN,NaN,True,False,True,True,True,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False
3,"250,000.00",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.00,1.00,True,1st Floor,NaN,<NA>,NaT,<NA>,True,<NA>,2.00,NaN,NaN,<NA>,114.00,NaN,NaN,0.00,"2,192.98",NaN,NaN,NaN,True,False,True,True,True,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False
4,"250,000.00",Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.00,1.00,True,2nd Floor,NaN,<NA>,NaT,<NA>,True,<NA>,2.00,NaN,NaN,<NA>,114.00,NaN,NaN,2.00,"2,192.98",NaN,NaN,NaN,True,False,True,True,True,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False


## Validação da Versão Preparada

A versão preparada deve cumprir critérios mínimos antes de ser gravada: sem duplicados exatos, sem valores em falta na variável alvo e sem valores objetivamente inválidos nas variáveis principais.


In [8]:
validation_checks = {
    "sem_price_missing": int(prepared["price"].isna().sum()) == 0,
    "sem_duplicados_exatos": int(prepared.duplicated().sum()) == 0,
    "sem_price_nao_positivo": int((prepared["price"] <= 0).sum()) == 0,
    "sem_total_area_nao_positiva": int((prepared["total_area"].dropna() <= 0).sum()) == 0 if "total_area" in prepared.columns else True,
    "sem_inf_price_m2": not np.isinf(prepared["price_m2"].dropna()).any() if "price_m2" in prepared.columns else True,
}

validation_table = pd.DataFrame(
    {
        "validacao": validation_checks.keys(),
        "resultado": validation_checks.values(),
    }
)

display(validation_table)

if not all(validation_checks.values()):
    failed = [key for key, value in validation_checks.items() if not value]
    raise ValueError(f"Validações falharam: {failed}")


,validacao,resultado
0,sem_price_missing,True
1,sem_duplicados_exatos,True
2,sem_price_nao_positivo,True
3,sem_total_area_nao_positiva,True
4,sem_inf_price_m2,True


## Variáveis com Risco de Fuga de Informação

A variável `price_m2` é útil para análise exploratória, mas é derivada diretamente de `price`. Por isso, não deve ser usada como preditor direto em modelos cujo objetivo seja prever `price`.

Nesta fase a variável é mantida no dataset preparado para diagnóstico, mas deve ser excluída nas pipelines de modelação.


In [9]:
modeling_excluded_features = ["price_m2"]
modeling_candidate_features = [
    col for col in prepared.columns
    if col not in ["price", *modeling_excluded_features]
]

feature_policy = pd.DataFrame(
    {
        "grupo": ["target", "excluir_modelacao", "candidatas_modelacao"],
        "variaveis": ["price", ", ".join(modeling_excluded_features), len(modeling_candidate_features)],
    }
)

display(feature_policy)


,grupo,variaveis
0,target,price
1,excluir_modelacao,price_m2
2,candidatas_modelacao,47


## Gravação do Dataset Preparado

A versão preparada é gravada numa localização separada da versão raw e da versão processada inicial.

São criados dois ficheiros:

- `portugal_listings_prepared.csv`, útil para trabalho local e inspeção direta;
- `portugal_listings_prepared.csv.gz`, versão comprimida recomendada para upload no GitHub.

Em ambiente Kaggle, estes ficheiros ficam em `/kaggle/working/` e aparecem na área de outputs do notebook. O Kaggle não envia automaticamente ficheiros para o GitHub; o ficheiro recomendado para download/upload é o `.csv.gz`, porque fica abaixo do limite de 25 MB do upload web do GitHub.


In [10]:
PREPARED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
prepared.to_csv(PREPARED_DATA_PATH, index=False)
prepared.to_csv(PREPARED_COMPRESSED_PATH, index=False, compression="gzip")

csv_size_mb = PREPARED_DATA_PATH.stat().st_size / (1024 ** 2)
gz_size_mb = PREPARED_COMPRESSED_PATH.stat().st_size / (1024 ** 2)

print(f"Dataset preparado guardado em: {PREPARED_DATA_PATH}")
print(f"Versão comprimida guardada em: {PREPARED_COMPRESSED_PATH}")
print(f"Linhas: {prepared.shape[0]:,}")
print(f"Colunas: {prepared.shape[1]:,}")
print(f"Tamanho CSV: {csv_size_mb:.1f} MB")
print(f"Tamanho CSV.GZ: {gz_size_mb:.1f} MB")


Dataset preparado guardado em: /kaggle/working/portugal_listings_prepared.csv
Versão comprimida guardada em: /kaggle/working/portugal_listings_prepared.csv.gz
Linhas: 126,242
Colunas: 49
Tamanho CSV: 31.5 MB
Tamanho CSV.GZ: 4.3 MB


## Conclusão

Este notebook criou uma versão preparada do dataset imobiliário português, preservando os dados originais e mantendo as decisões de limpeza explícitas e auditáveis.

Principais pontos a reter:
- os duplicados exatos e valores sem `price` são controlados na versão preparada;
- regras objetivas de domínio são aplicadas a áreas, contagens, ano de construção e variáveis derivadas;
- valores em falta relevantes são sinalizados com indicadores próprios;
- outliers são assinalados por IQR, mas não removidos automaticamente;
- `price_m2` deve ser tratado como variável de análise e excluído como preditor direto de `price`.

Artefacto produzido:
- `data/processed/portugal_listings_prepared.csv`, ou `/kaggle/working/portugal_listings_prepared.csv` em ambiente Kaggle.
- `data/processed/portugal_listings_prepared.csv.gz`, ou `/kaggle/working/portugal_listings_prepared.csv.gz`, como versão comprimida recomendada para GitHub.

Próxima etapa:
- avançar para feature engineering, definindo variáveis candidatas, transformações, encoding e prevenção formal de fuga de informação antes da modelação.

**Última alteração:** 18/06/2026
